# Ingestão e camada Bronze

**Tech Challenge Fase 3** · Pós-Tech em Data Analytics, FIAP
**Base:** State of Data Brazil, edições 2023-2024, 2024-2025 e 2025-2026
**Etapa do pipeline:** ingestão dos arquivos de origem no data lake

A camada Bronze recebe os arquivos exatamente como saíram do Kaggle, sem nenhuma alteração. Qualquer correção de acentuação, tipo ou formato feita antes deste ponto descaracterizaria a camada, que existe justamente para ser fiel à origem.

## 1. Configuração da sessão Spark

A sessão local reproduz o mesmo comportamento do AWS Glue. Ao portar o código para o Glue, apenas a linha `.master(...)` é removida e os caminhos locais passam a apontar para o S3.

In [1]:
import sys, os
os.environ.pop("JAVA_TOOL_OPTIONS", None)
sys.path.insert(0, "../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("tc3")
    .master("local[2]")                      # no AWS Glue esta linha nao existe
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

26/09/05 21:52:49 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/09/05 21:52:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/05 21:52:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3


## 2. Conferência dos arquivos recebidos

Antes de qualquer processamento, a estrutura de cada arquivo é verificada. Os três correspondem às três últimas edições publicadas pelo Data Hackers, exigência da página 3 do enunciado.

In [2]:
from schema import ler_cabecalhos, EDICOES

cabecalhos = ler_cabecalhos("../dados/raw")
for edicao, linhas in cabecalhos.items():
    print(f"{edicao}: {len(linhas)} colunas")

2023-2024: 399 colunas
2024-2025: 403 colunas
2025-2026: 388 colunas


## 3. Leitura dos dados brutos

A leitura usa `multiLine` e `escape` porque várias respostas abertas contêm quebras de linha e aspas.

In [3]:
bronze = {}
for edicao, arquivo in EDICOES.items():
    df = (
        spark.read.option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .option("encoding", "UTF-8")
        .csv(f"../dados/raw/{arquivo}")
    )
    bronze[edicao] = df
    print(f"{edicao}: {df.count()} linhas, {len(df.columns)} colunas")

2023-2024: 5293 linhas, 399 colunas


2024-2025: 5217 linhas, 403 colunas


2025-2026: 3495 linhas, 388 colunas


## 4. Organização no data lake

No AWS Academy Lab, os arquivos são enviados ao S3 na estrutura abaixo, particionada por edição, o que permite consultar uma edição isolada sem varrer as demais.

```
s3://<bucket>/bronze/edicao=2023-2024/
s3://<bucket>/bronze/edicao=2024-2025/
s3://<bucket>/bronze/edicao=2025-2026/
```

In [4]:
# verificacao de integridade antes de seguir para a camada Silver
for edicao, df in bronze.items():
    total = df.count()
    distintas = df.dropDuplicates().count()
    print(f"{edicao}: {total} linhas, {total - distintas} duplicata(s) exata(s)")

2023-2024: 5293 linhas, 0 duplicata(s) exata(s)


2024-2025: 5217 linhas, 2 duplicata(s) exata(s)


2025-2026: 3495 linhas, 1 duplicata(s) exata(s)


**Leitura do resultado.** As edições 2024-2025 e 2025-2026 trazem, respectivamente, duas e uma linha integralmente duplicadas. São duplicações reais de registro, não colisão de identificador, portanto podem ser removidas com segurança na camada Silver.